In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
#import cartopy.crs as ccrs
import xesmf as xe
from dask.diagnostics import ProgressBar

In [ ]:
GEBCO_SWA_ds = xr.open_dataset("./resources/GEBCO/gebco_2025_n-32.45_s-60.0_w-69.61_e-50.0.nc")

In [ ]:
GEBCO_SWA_ds

<xarray.Dataset> Size: 62MB
Dimensions:    (lat: 6612, lon: 4706)
Coordinates:
  * lat        (lat) float64 53kB -60.0 -59.99 -59.99 ... -32.46 -32.46 -32.45
  * lon        (lon) float64 38kB -69.61 -69.6 -69.6 ... -50.01 -50.01 -50.0
Data variables:
    elevation  (lat, lon) int16 62MB ...
Attributes: (12/36)
    title:                           The GEBCO_2025 Grid - a continuous terra...
    summary:                         The GEBCO_2025 Grid is a continuous, glo...
    keywords:                        BATHYMETRY/SEAFLOOR TOPOGRAPHY, DIGITAL ...
    Conventions:                     CF-1.6, ACDD-1.3
    id:                              DOI: 10.5285/37c52e96-24ea-67ce-e063-708...
    naming_authority:                https://dx.doi.org
    ...                              ...
    geospatial_vertical_units:       meters
    geospatial_vertical_resolution:  1.0
    geospatial_vertical_positive:    up
    identifier_product_doi:          DOI: 10.5285/37c52e96-24ea-67ce-e063-708...
    references:                      DOI: 10.5285/37c52e96-24ea-67ce-e063-708...
    node_offset:                     1.0

In [ ]:
lat_diff = np.diff(GEBCO_SWA_ds.lat)
lon_diff = np.diff(GEBCO_SWA_ds.lon)
print("Latitude spacing unique values:", np.unique(lat_diff))
print("Longitude spacing unique values:", np.unique(lon_diff))

Latitude spacing unique values: [0.00416667 0.00416667 0.00416667]
Longitude spacing unique values: [0.00416667 0.00416667 0.00416667]


In [ ]:
print(GEBCO_SWA_ds.lon.ndim, GEBCO_SWA_ds.lat.ndim)  # should both be 1
print(GEBCO_SWA_ds.lon.shape, GEBCO_SWA_ds.lat.shape) 

1 1
(4706,) (6612,)


In [ ]:
#first coarsen to allow faster regridding
downsized_factor = 20
GEBCO_SWA_ds_coarse = GEBCO_SWA_ds.coarsen(lat=downsized_factor, lon=downsized_factor, boundary="trim").mean()

#compare original and coarsened
print("original size", GEBCO_SWA_ds.sizes)
print("coarsened size", GEBCO_SWA_ds_coarse.sizes)
lat_diff = np.diff(GEBCO_SWA_ds_coarse.lat)
lon_diff = np.diff(GEBCO_SWA_ds_coarse.lon)
print("Latitude spacing unique values:", np.unique(lat_diff))
print("Longitude spacing unique values:", np.unique(lon_diff))
print("Coarsened bounds:")
print(GEBCO_SWA_ds_coarse.lat.min().item(), GEBCO_SWA_ds_coarse.lat.max().item())
print(GEBCO_SWA_ds_coarse.lon.min().item(), GEBCO_SWA_ds_coarse.lon.max().item())
print("Original bounds:")
print(GEBCO_SWA_ds.lat.min().item(), GEBCO_SWA_ds.lat.max().item())
print(GEBCO_SWA_ds.lon.min().item(), GEBCO_SWA_ds.lon.max().item())


original size Frozen({'lat': 6612, 'lon': 4706})
coarsened size Frozen({'lat': 330, 'lon': 235})
Latitude spacing unique values: [0.08333333 0.08333333 0.08333333]
Longitude spacing unique values: [0.08333333 0.08333333 0.08333333 0.08333333 0.08333333 0.08333333]
Coarsened bounds:
-59.958333333333336 -32.54166666666667
-69.56666666666668 -50.06666666666668
Original bounds:
-59.99791666666667 -32.452083333333334
-69.60625 -50.00208333333333


In [ ]:
#now regrid to regular grid for the model
min_lon, min_lat, max_lon, max_lat = [-69.61, -60., -50., -32.45] 

# Target lat/lon vectors for a resolution of 0.125 degrees
res = 0.125# Target resolution (degrees)
new_lons = np.arange(min_lon, max_lon + res, res)
new_lats = np.arange(min_lat, max_lat + res, res)

# Create a target grid as xarray Dataset
ds_tgt = xr.Dataset({
    'lat': (['lat'], new_lats),
    'lon': (['lon'], new_lons)})

regridder = xe.Regridder(GEBCO_SWA_ds_coarse, ds_tgt, 'conservative')
GEBCO_SWA_regrided = regridder(GEBCO_SWA_ds_coarse)


In [ ]:
#compare original and regrided
print("original size", GEBCO_SWA_ds.sizes)
print("regrided size", GEBCO_SWA_regrided.sizes)
lat_diff = np.diff(GEBCO_SWA_regrided.lat)
lon_diff = np.diff(GEBCO_SWA_regrided.lon)
print("Regrided spacing:")
print("Lat spacing:", np.unique(lat_diff), "Lon spacing:", np.unique(lon_diff))
print("Original bounds:")
print(GEBCO_SWA_ds.lat.min().item(), GEBCO_SWA_ds.lat.max().item())
print(GEBCO_SWA_ds.lon.min().item(), GEBCO_SWA_ds.lon.max().item())
print("Regrided bounds:")
print(GEBCO_SWA_regrided.lat.min().item(), GEBCO_SWA_regrided.lat.max().item())
print(GEBCO_SWA_regrided.lon.min().item(), GEBCO_SWA_regrided.lon.max().item())


original size Frozen({'lat': 6612, 'lon': 4706})
regrided size Frozen({'lat': 222, 'lon': 158})
Regrided spacing:
Lat spacing: [0.125] Lon spacing: [0.125]
Original bounds:
-59.99791666666667 -32.452083333333334
-69.60625 -50.00208333333333
Regrided bounds:
-60.0 -32.375
-69.61 -49.985
